<a href="https://colab.research.google.com/github/JOk3r01001/Fracture-detection-imagenett---focal_loss/blob/main/Keras_weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import drive



drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import pandas as pd
import shutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB1
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import matplotlib.pyplot as plt

# Paths and parameters
BASE_PATH   = "/content/drive/MyDrive/BK2.0/FracAtlas2/images"
LOCAL_KERAS = "/content/keras_data"
IMG_SIZE    = (640, 640)
BATCH_SIZE  = 8
SEED        = 42

# 1. Build dataframe directly from folder structure
records = []
for label_name, label_val in [("Fractured", 1), ("Non_fractured", 0)]:
    folder = os.path.join(BASE_PATH, label_name)
    for fname in os.listdir(folder):
        if fname.lower().endswith(('.jpg')):
            records.append({
                "img_name": fname,
                "filepath": os.path.join(folder, fname),
                "label":    label_val,
                "label_str": str(label_val)
            })

df = pd.DataFrame(records)
print(f"Total images: {len(df)}  |  Fractured: {df['label'].sum()}  |  Non_fractured: {(df['label']==0).sum()}")

# 2. Detect and remove corrupted images
def remove_corrupted(df, name):
    bad = []
    for idx, row in df.iterrows():
        try:
            with Image.open(row['filepath']) as img:
                img.verify()
        except Exception:
            bad.append(idx)
            print(f"  Corrupted: {row['img_name']}")
    print(f"{name}: removed {len(bad)} corrupted images")
    return df.drop(bad).reset_index(drop=True)

df = remove_corrupted(df, "ALL")

# 3. Stratified split 80 / 10 / 10
train_df, tmp_df = train_test_split(df,     test_size=0.20, stratify=df['label'],     random_state=SEED)
val_df,  test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df['label'], random_state=SEED)

print(f"\nAfter split:")
print(f"  Train : {len(train_df)}  (frac={train_df['label'].mean():.2f})")
print(f"  Val   : {len(val_df)}   (frac={val_df['label'].mean():.2f})")
print(f"  Test  : {len(test_df)}  (frac={test_df['label'].mean():.2f})")

# Copy images to local environment for faster data loading
for split in ['train', 'val', 'test']:
    for cls in ['Fractured', 'Non_fractured']:
        os.makedirs(os.path.join(LOCAL_KERAS, split, cls), exist_ok=True)

def copy_to_local(df, split_name):
    print(f"Copying {split_name}...")
    for _, row in df.iterrows():
        folder = 'Fractured' if row['label'] == 1 else 'Non_fractured'
        dst = os.path.join(LOCAL_KERAS, split_name, folder, row['img_name'])
        if not os.path.exists(dst):
            shutil.copy(row['filepath'], dst)
    print(f"  Done: {len(df)} images")

copy_to_local(train_df, 'train')
copy_to_local(val_df,   'val')
copy_to_local(test_df,  'test')

# Update file paths to local environment
def local_path(row, split):
    folder = 'Fractured' if row['label'] == 1 else 'Non_fractured'
    return os.path.join(LOCAL_KERAS, split, folder, row['img_name'])

train_df['filepath'] = train_df.apply(lambda r: local_path(r, 'train'), axis=1)
val_df['filepath']   = val_df.apply(lambda r: local_path(r, 'val'),   axis=1)
test_df['filepath']  = test_df.apply(lambda r: local_path(r, 'test'),  axis=1)

print("\nAll data copied locally")

# Data generators with augmentation for training set
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    horizontal_flip=True,
    rotation_range=15,
    fill_mode='nearest'
)
# Validation and test generators - preprocessing only
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

def make_gen(datagen, df, shuffle):
    return datagen.flow_from_dataframe(
        df,
        x_col='filepath',
        y_col='label_str',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary',
        classes=['0', '1'],
        shuffle=shuffle
    )

train_gen = make_gen(train_datagen, train_df, shuffle=True)
val_gen   = make_gen(val_datagen,   val_df,   shuffle=False)
test_gen  = make_gen(val_datagen,   test_df,  shuffle=False)

print("\nGenerators are ready")
print(f"  Train batches : {len(train_gen)}")
print(f"  Val batches   : {len(val_gen)}")
print(f"  Test batches  : {len(test_gen)}")

Celkem obrázků: 4083  |  Fractured: 717  |  Non_fractured: 3366
ALL: odstraněno 0 poškozených

Po rozdělení:
  Train : 3266  (frac=0.18)
  Val   : 408   (frac=0.17)
  Test  : 409  (frac=0.18)
Copying train...
  Done: 3266 images
Copying val...
  Done: 408 images
Copying test...
  Done: 409 images

Všechna data zkopírována lokálně!
Found 3266 validated image filenames belonging to 2 classes.
Found 408 validated image filenames belonging to 2 classes.
Found 409 validated image filenames belonging to 2 classes.

Generátory připraveny!
  Train batches : 409
  Val batches   : 51
  Test batches  : 52


In [4]:
import os
from google.colab import drive

# Mount Google Drive to access files
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB1
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from PIL import Image

# Output directory for model weights and logs
PROJECT_OUT = "/content/drive/MyDrive/KerasNET_vahy"
os.makedirs(PROJECT_OUT, exist_ok=True)

# Load pretrained EfficientNetB1 backbone without classification head
backbone = EfficientNetB1(
    include_top=False,
    weights='imagenet',
    input_shape=(640, 640, 3)
)
backbone.trainable = True

# Define custom classification head
inputs = keras.Input(shape=(640, 640, 3))
x = backbone(inputs, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)

# Focal loss - focuses training on hard misclassified examples
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = alpha * tf.pow(1 - p_t, gamma)
        return tf.reduce_mean(focal_weight * bce)
    return loss

# Compile model
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.0005),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

model.summary()

# Callbacks for training control
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_recall',
        patience=10,
        mode='max',
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(PROJECT_OUT, 'efficientnet_best.keras'),
        monitor='val_recall',
        save_best_only=True,
        mode='max'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7
    ),
    keras.callbacks.CSVLogger(
        os.path.join(PROJECT_OUT, 'efficientnet_training_log.csv')
    )
]

# Train model on training data with validation
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=callbacks,
)

# Evaluate model on test set
print("\nTest set evaluation:")
results = model.evaluate(test_gen)
print(f"Accuracy:  {results[1]:.4f}")
print(f"Precision: {results[2]:.4f}")
print(f"Recall:    {results[3]:.4f}")
print(f"AUC:       {results[4]:.4f}")

test_gen.reset()
preds = (model.predict(test_gen) > 0.5).astype(int)
print("\nClassification Report:")
print(classification_report(
    test_gen.classes,
    preds,
    target_names=['No Fracture', 'Fracture']
))

# Confusion matrix to visualize correct and incorrect classifications
cm = confusion_matrix(test_gen.classes, preds)
print("\nConfusion Matrix:")
print(cm)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 640, 640, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb1 (Functional)     │ (None, 20, 20, 1280)   │     6,575,239 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,941,320 (26.48 MB)

 Trainable params: 6,876,705 (26.23 MB)

 Non-trainable params: 64,615 (252.41 KB)

Epoch 1/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 707s 1s/step - accuracy: 0.7538 - auc: 0.6168 - loss: 0.0659 - precision: 0.2932 - recall: 0.2840 - val_accuracy: 0.8333 - val_auc: 0.7369 - val_loss: 0.0278 - val_precision: 0.5349 - val_recall: 0.3239 - learning_rate: 5.0000e-04
Epoch 2/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 460s 1s/step - accuracy: 0.8129 - auc: 0.7398 - loss: 0.0316 - precision: 0.4602 - recall: 0.3728 - val_accuracy: 0.8260 - val_auc: 0.8017 - val_loss: 0.0260 - val_precision: 0.5000 - val_recall: 0.5211 - learning_rate: 5.0000e-04
Epoch 3/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 500s 1s/step - accuracy: 0.8362 - auc: 0.7911 - loss: 0.0272 - precision: 0.5450 - recall: 0.4111 - val_accuracy: 0.8775 - val_auc: 0.8347 - val_loss: 0.0212 - val_precision: 0.7333 - val_recall: 0.4648 - learning_rate: 5.0000e-04
Epoch 4/50
409/409 ━━━━━━━━━━━━━━━━━━━━ 461s 1s/step - accuracy: 0.8515 - auc: 0.8082 - loss: 0.0254 - precision: 0.6023 - recall: 0.4564 - val_accuracy: 0.8897 - val_auc: 0.8240 - val_lo

In [5]:
from sklearn.metrics import precision_recall_curve, roc_curve

# Získání pravděpodobností
test_gen.reset()
y_proba = model.predict(test_gen).flatten()
y_true  = test_gen.classes

# Testování různých thresholdů
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Accuracy':<12}")
print("-" * 60)
for t in thresholds:
    y_pred = (y_proba > t).astype(int)
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
    print(f"{t:<12.2f} {precision_score(y_true, y_pred):<12.4f} {recall_score(y_true, y_pred):<12.4f} {f1_score(y_true, y_pred):<12.4f} {accuracy_score(y_true, y_pred):<12.4f}")

52/52 ━━━━━━━━━━━━━━━━━━━━ 8s 147ms/step
Threshold    Precision    Recall       F1           Accuracy    
------------------------------------------------------------
0.30         0.2834       0.9722       0.4389       0.5623      
0.40         0.4599       0.8750       0.6029       0.7971      
0.50         0.7143       0.7639       0.7383       0.9046      
0.60         0.8246       0.6528       0.7287       0.9144      
0.70         0.9535       0.5694       0.7130       0.9193      
0.80         1.0000       0.4583       0.6286       0.9046      
0.90         1.0000       0.2639       0.4176       0.8704      
